In [ ]:
# =========================
# nb_fuzzy_resolution
# =========================
# Phase 9, SC-13: Use AI to suggest ISIN mappings for unmapped securities
# Leverages shared_ai_utils.py for GitHub Models or Azure OpenAI backend

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
import sys
import json

# ---------- Pre-flight checks ----------
try:
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    print(f"[nb_fuzzy_resolution] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

if not spark.catalog.tableExists("validation_events") or not spark.catalog.tableExists("ai_resolution_suggestions"):
    print("[ERROR] Required tables do not exist")
    sys.exit(1)

# ---------- Load shared_ai_utils ----------
sys.path.append('/lakehouse/default/Files/config')
try:
    from shared_ai_utils import load_ai_config, AIClient
    ai_enabled = True
except ImportError as e:
    print(f"[WARNING] shared_ai_utils not found: {e}")
    print("[nb_fuzzy_resolution] AI not available; skipping")
    mssparkutils.notebook.exit("AI_DISABLED")
except Exception as e:
    print(f"[ERROR] Failed to import shared_ai_utils: {type(e).__name__}: {e}")
    sys.exit(1)

# Load AI config
try:
    ai_client = load_ai_config('/lakehouse/default/Files/config/azure_openai_config.json')
    if ai_client is None:
        raise RuntimeError("AI config returned None")
except Exception as e:
    print(f"[ERROR] Failed to load AI config: {type(e).__name__}: {e}")
    mssparkutils.notebook.exit("CONFIG_ERROR")

# ---------- Load data ----------
try:
    ch = spark.table("canonical_holdings").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    if ch.rdd.isEmpty():
        print(f"[nb_fuzzy_resolution] No canonical_holdings for period={period}, run_id={run_id}")
        mssparkutils.notebook.exit("NO_DATA")
    
    # Find unmapped securities (null ISIN or MAP_001 validation failures)
    ve = spark.table("validation_events").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id) &
        (F.col("rule_id") == "MAP_001") & (F.col("status") == "fail")
    )
    
    if ve.rdd.isEmpty():
        print("[nb_fuzzy_resolution] No unmapped securities found (MAP_001 failures)")
        mssparkutils.notebook.exit("OK")
    
    unmapped = ve.select("dfm_id", "policy_id", "security_id").distinct()
    unmapped_count = unmapped.count()
    print(f"[nb_fuzzy_resolution] Processing {unmapped_count} unmapped securities...")
    
except Exception as e:
    print(f"[ERROR] Failed to load data: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

# ---------- AI resolution ----------
suggestions = []
success_count = 0
error_count = 0

try:
    for row in unmapped.collect():
        dfm_id = str(row["dfm_id"]) if row["dfm_id"] else "UNKNOWN"
        policy_id = str(row["policy_id"]) if row["policy_id"] else "UNKNOWN"
        security_id = str(row["security_id"]) if row["security_id"] else "UNKNOWN"
        
        # Build prompt for AI
        prompt = f"""
Given the security identifier: {security_id}
For policy holder: {policy_id}
From DFM: {dfm_id}

Suggest the top 3 possible ISINs or security codes that this identifier might map to.
Provide confidence scores (0.0-1.0) and reasoning for each suggestion.

Return JSON with format: {{
  "candidates": [
    {{"isin": "XX...", "name": "...", "confidence": 0.95, "reasoning": "..."}}
  ]
}}
"""
        
        try:
            result_text = ai_client.schema_map(prompt)
            
            # Validate response structure
            if not result_text or not isinstance(result_text, str):
                raise ValueError("AI returned non-string response")
            
            result = json.loads(result_text)
            
            if "candidates" not in result or not isinstance(result["candidates"], list):
                raise ValueError("AI response missing 'candidates' array")
            
            # Process candidates
            candidates_added = 0
            for idx, candidate in enumerate(result.get("candidates", [])[:3]):
                if not isinstance(candidate, dict):
                    continue
                
                try:
                    suggestions.append({
                        "period": period,
                        "run_id": run_id,
                        "dfm_id": dfm_id,
                        "policy_id": policy_id,
                        "security_id": security_id,
                        "unmapped_value": security_id,
                        "candidate_isin": str(candidate.get("isin", "")),
                        "candidate_name": str(candidate.get("name", "")),
                        "confidence_score": float(candidate.get("confidence", 0.0)),
                        "rank": idx + 1,
                        "reasoning": str(candidate.get("reasoning", "")),
                        "ai_model": "gpt-4o",
                        "suggested_action": "manual_review" if idx > 0 else "accept",
                        "resolved_at": None
                    })
                    candidates_added += 1
                except (ValueError, TypeError) as e:
                    print(f"[WARNING] Failed to parse candidate for {security_id}: {e}")
                    error_count += 1
            
            if candidates_added > 0:
                success_count += 1
            else:
                error_count += 1
                print(f"[WARNING] No valid candidates for {security_id}")
        
        except json.JSONDecodeError as e:
            error_count += 1
            print(f"[WARNING] AI response not valid JSON for {security_id}: {e}")
        except Exception as e:
            error_count += 1
            print(f"[WARNING] Error processing {security_id}: {type(e).__name__}: {e}")

    # Write results
    if suggestions:
        try:
            df_suggestions = spark.createDataFrame(suggestions)
            df_suggestions.write.mode("append").saveAsTable("ai_resolution_suggestions")
            print(f"[nb_fuzzy_resolution] COMPLETE wrote {len(suggestions)} suggestions (success={success_count}, skipped={error_count})")
        except Exception as e:
            print(f"[ERROR] Failed to write suggestions: {type(e).__name__}: {e}")
            sys.exit(1)
    else:
        print(f"[WARNING] No valid suggestions generated (success={success_count}, errors={error_count})")
    
    mssparkutils.notebook.exit("OK")

except Exception as e:
    print(f"[ERROR] AI resolution failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)